### Read experimental result data

In [1]:
# read parquet file
import polars as pl
# df = pl.read_parquet("../../experiments/16_10_result.parquet")
df = pl.read_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_DE.parquet")
print(df)

# read metadata of the parquet file
import pyarrow.parquet as pq
# meta = pq.read_metadata("../test/result.parquet")
meta = pq.read_metadata("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_DE.parquet")

# You can check versions depending on libraries 
print(meta.metadata)

shape: (524_288, 5)
┌─────┬──────┬───────┬───────┬───────────────┐
│ id  ┆ ip   ┆ us_id ┆ msg   ┆ mean_num_xact │
│ --- ┆ ---  ┆ ---   ┆ ---   ┆ ---           │
│ u64 ┆ u64  ┆ u64   ┆ str   ┆ f64           │
╞═════╪══════╪═══════╪═══════╪═══════════════╡
│ 0   ┆ 9205 ┆ 0     ┆ 30000 ┆ 26.61         │
│ 0   ┆ 9205 ┆ 0     ┆ 20000 ┆ 20.92         │
│ 0   ┆ 9205 ┆ 0     ┆ 31000 ┆ 27.14         │
│ 0   ┆ 9205 ┆ 0     ┆ 21000 ┆ 29.21         │
│ 0   ┆ 9205 ┆ 0     ┆ 01000 ┆ 26.63         │
│ …   ┆ …    ┆ …     ┆ …     ┆ …             │
│ 511 ┆ 2821 ┆ 31    ┆ 32333 ┆ 23.19         │
│ 511 ┆ 2821 ┆ 31    ┆ 33333 ┆ 22.49         │
│ 511 ┆ 2821 ┆ 31    ┆ 03333 ┆ 47.31         │
│ 511 ┆ 2821 ┆ 31    ┆ 13333 ┆ 27.95         │
│ 511 ┆ 2821 ┆ 31    ┆ 23333 ┆ 29.95         │
└─────┴──────┴───────┴───────┴───────────────┘
{b'version_runner': b'0.1.1', b'version_core': b'0.1.1', b'version': b'0.1.0', b'ARROW:schema': b'/////0wBAAAQAAAAAAAKAAwACgAJAAQACgAAABAAAAAAAQQACAAIAAAABAAIAAAABAAAAAUAAADsAAAAsAA

### Compute solutions and/or values of optimal, RAM, and URS policies

In [2]:
EV_BY_OPT = "ev_by_opt"
EV_BY_RAM = "ev_by_ram"
EV_BY_URS = "ev_by_urs"
OPT_MEAN = "mean_of_opt_values"
RAMEV_OPTEV_RATIO = "ram ev / opt ev"
URSEV_OPTEV_RATIO = "urs ev / opt ev"
RAMEV_OPTM_RATIO = "ram ev / opt mean"
URSEV_OPTM_RATIO = "urs ev / opt mean"

MEAN_NUM_XACT = "mean_num_xact"

lf_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["us_id", "ip"]))
    .unique(["ip", "us_id", MEAN_NUM_XACT])
    .select([pl.col("us_id"), pl.col("ip"), pl.col("msg").name.prefix("argmax_"), pl.col(MEAN_NUM_XACT).name.prefix("max_")])
)
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))
    .group_by("ip_right", "us_id_right")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])
    .select([MEAN_NUM_XACT, "max_" + MEAN_NUM_XACT])
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias(EV_BY_OPT), pl.col("max_" + MEAN_NUM_XACT).alias(OPT_MEAN)])
)
lf_rob = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max())
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_RAM))
)
lf_urs = (
    df.lazy()
    .select(MEAN_NUM_XACT)
    .mean()
    .select(pl.col(MEAN_NUM_XACT).alias(EV_BY_URS))
)
eval_df = (
    pl.concat([lf_opt_mod, lf_rob, lf_urs], how="horizontal")
    .with_columns([
        (pl.col(EV_BY_RAM) / pl.col(EV_BY_OPT)).alias(RAMEV_OPTEV_RATIO),
        (pl.col(EV_BY_URS) / pl.col(EV_BY_OPT)).alias(URSEV_OPTEV_RATIO),
        (pl.col(EV_BY_RAM) / pl.col(OPT_MEAN)).alias(RAMEV_OPTM_RATIO),
        (pl.col(EV_BY_URS) / pl.col(OPT_MEAN)).alias(URSEV_OPTM_RATIO),
    ])
).collect().transpose(include_header=True).rename({"column": "key", "column_0": "value"})

In [3]:
lf_opt_mod = (df.lazy()
    .join(lf_opt, left_on=pl.col("msg"), right_on=pl.col("argmax_msg"))#普通のdfにlf_optで事後確認した最適なmsgを他にip,us_idに適用されたのがあればそれをあわせて平均化。
    .group_by("ip_right", "us_id_right")#他のip,user_stateにもoptのmsgが通用するかという問い
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .join(lf_opt, left_on=["ip_right", "us_id_right"], right_on=["ip", "us_id"])#ipとus_idにピッタリなmsg
    .mean()
    .select([pl.col(MEAN_NUM_XACT).alias("ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値"),
             pl.col("max_" + MEAN_NUM_XACT).alias("ipとuser_stateに最適なmsg適用後の期待値")])
)
lf_opt_mod.mean().collect()

"ip,user_state関係なく、最適なmsgをランダムに適用したxact期待値",ipとuser_stateに最適なmsg適用後の期待値
f64,f64
54.055612,81.966211


ipだけぴったりのmsgだけ適用すると?

In [4]:
ip_opt = (
    df.lazy()
    .filter(pl.col(MEAN_NUM_XACT) == pl.col(MEAN_NUM_XACT).max().over(["ip"]))
    .unique(["ip", MEAN_NUM_XACT])
    .select([pl.col("ip"), pl.col("msg"), pl.col(MEAN_NUM_XACT)])
).collect()


In [5]:
ip_opt

ip,msg,mean_num_xact
u64,str,f64
2821,"""00003""",604.54
5603,"""00023""",31.22
4236,"""03032""",56.42
5551,"""00122""",85.06
7536,"""00033""",1167.31
…,…,…
9457,"""00133""",4.8
2872,"""00023""",217.86
6257,"""10020""",5.62


In [6]:
ip_opt.select(pl.col("mean_num_xact").mean().alias("ip_opt_msg_xact"))#平均値がipとuser_stateより高い

ip_opt_msg_xact
f64
184.383125


In [7]:
ip_opt.group_by("msg").len()

msg,len
str,u32
"""00000""",2
"""03032""",1
"""00133""",2
"""00033""",1
"""10000""",1
…,…
"""00003""",2
"""00013""",1
"""10020""",1


In [8]:
ip_opt_list=(ip_opt.unique("msg").select(pl.col("msg")))["msg"].to_list()

In [9]:
ip_opt_list

['00122',
 '00013',
 '10023',
 '10020',
 '00133',
 '03032',
 '00033',
 '00003',
 '10000',
 '00023',
 '00000',
 '32333']

ipだけぴったりのmsgだけ適用すると外部行動者数が増加した

In [10]:
ip_opt_mod = (df.lazy()#ipにピッタリなmsgを他のシナリオに適用した場合。
    .join(ip_opt.lazy(), left_on=pl.col("msg"), right_on=pl.col("msg"))
    .group_by("ip_right").agg(pl.col("mean_num_xact").mean()).sort("mean_num_xact",descending=True)
    .select(pl.col("ip_right").alias("ip"),pl.col("mean_num_xact").alias("他のシナリオにmsgを適用したxact"))).collect()
ip_opt_frame=ip_opt_mod.join(ip_opt,left_on="ip",right_on="ip").select(pl.col("ip"),pl.col("msg"),pl.col("他のシナリオにmsgを適用したxact")).sort("他のシナリオにmsgを適用したxact",descending = True)
ip_opt_frame

ip,msg,他のシナリオにmsgを適用したxact
u64,str,f64
2821,"""00003""",77.641016
5631,"""00003""",77.641016
4752,"""00013""",77.032598
5603,"""00023""",75.174082
2872,"""00023""",75.174082
…,…,…
6257,"""10020""",35.372754
3905,"""00000""",31.671895
2606,"""00000""",31.671895


In [11]:
ip_opt_frame.select(pl.col("他のシナリオにmsgを適用したxact").mean().alias("他のシナリオにmsgを適用したxactの期待値"))

他のシナリオにmsgを適用したxactの期待値
f64
56.082524


In [12]:
eval_df

key,value
str,f64
"""ev_by_opt""",54.055612
"""mean_of_opt_values""",81.966211
"""ev_by_ram""",77.641016
"""ev_by_urs""",33.203325
"""ram ev / opt ev""",1.436317
"""urs ev / opt ev""",0.614244
"""ram ev / opt mean""",0.947232
"""urs ev / opt mean""",0.405086


In [57]:
###ram
lf_rob_details = (
    df.lazy()
    .group_by("msg")
    .agg(pl.col(MEAN_NUM_XACT).mean())
    .sort(by="mean_num_xact", descending=True)
    # 上位5行を取得
).collect()
lf_rob_details.filter(pl.col("mean_num_xact")>=pl.col("mean_num_xact").mean()*1.8)

msg,mean_num_xact
str,f64
"""00003""",77.641016
"""00013""",77.032598
"""00023""",75.174082
"""00033""",72.45252
"""00103""",69.998125
…,…
"""20023""",60.775879
"""00112""",60.629648
"""00122""",60.389238


In [14]:
lf_rob_details.select(pl.col("mean_num_xact").mean())

mean_num_xact
f64
33.203325


In [15]:
# read parquet file
import polars as pl
# user_analysis= pl.read_parquet("../../experiments/user_analysis.parquet")
user_analysis= pl.read_parquet("/Users/shinomilab-mac-pro/seokyeong/diffusion_seokyeong/test/twitch_DE_user_analysis.parquet")
user_analysis


ip,us_id,msg,user_id,num_xact_of_users,num_share_of_users,distance_from_ip
u64,u64,str,u64,f64,f64,u64
9205,0,"""30000""",2,0.01,0.0,2
9205,0,"""30000""",8,0.36,0.0,3
9205,0,"""30000""",16,0.29,0.0,3
9205,0,"""30000""",21,0.04,0.0,3
9205,0,"""30000""",30,0.35,0.01,2
…,…,…,…,…,…,…
2821,31,"""23333""",9131,0.05,0.05,3
2821,31,"""23333""",9164,0.08,0.0,2
2821,31,"""23333""",9272,0.05,0.01,2


In [58]:
ram_msg = lf_rob_details.filter(pl.col("mean_num_xact")>=pl.col("mean_num_xact").mean()*1.8).select(pl.col("msg"))
ram_msg_list =ram_msg["msg"].to_list()

ram_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).collect()

# # reverse_msg_list = lf_rob_details.lazy().sort("mean_num_xact",descending=True).select(pl.col("msg"))[-10:].to_list()
# # reverse_msg_us_an = user_analysis.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).collect()


In [59]:
ram_msg_list

['00003',
 '00013',
 '00023',
 '00033',
 '00103',
 '10003',
 '10013',
 '00022',
 '00113',
 '10023',
 '00123',
 '00032',
 '00012',
 '01003',
 '01013',
 '10033',
 '20003',
 '01023',
 '00002',
 '00133',
 '20013',
 '10103',
 '00203',
 '01033',
 '10022',
 '20023',
 '00112',
 '00122',
 '10113',
 '10012']

In [ ]:
lf_opt.sort("max_mean_num_xact",descending=True)

us_id,ip,argmax_msg,max_mean_num_xact
u64,u64,str,f64
4,7536,"""00033""",1167.31
17,7536,"""00033""",1099.44
12,7536,"""10023""",1034.77
26,7536,"""20023""",983.25
13,7536,"""00013""",982.09
…,…,…,…
22,775,"""00000""",0.0
19,775,"""10000""",0.0
31,775,"""00000""",0.0


In [69]:
lf_opt_list=(lf_opt
             .sort("max_mean_num_xact",descending=True).filter(pl.col("max_mean_num_xact")>33.203325
*1.8).collect())

lf_opt_list.group_by("argmax_msg").len().sort("len",descending=True)["argmax_msg"].to_list()
# lf_opt.sort("max_mean_num_xact",descending=True).collect()[:25]

['00003',
 '00013',
 '10003',
 '00033',
 '00023',
 '10013',
 '20003',
 '10023',
 '01003',
 '20013',
 '01023',
 '30013',
 '00103',
 '20023',
 '00133',
 '01013',
 '10033',
 '00122',
 '00022']

In [19]:
ip_opt_list

['00122',
 '00013',
 '10023',
 '10020',
 '00133',
 '03032',
 '00033',
 '00003',
 '10000',
 '00023',
 '00000',
 '32333']

00003,00033,00013,00002,00012は共通
つまり、少なくとも3ラウンドまで内部行動の一番強のメッセージを連続に伝播して、徐々に外部行動強のメッセージを伝播するのが外部行動者数を最大化してくれる。

In [18]:
reverse_msg_list

['33230',
 '23331',
 '32330',
 '23320',
 '33333',
 '33332',
 '33331',
 '33320',
 '23330',
 '33330']

In [70]:
ratio_xact_share_per = (user_analysis.lazy()
.select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")).collect())
xact_per_mean = (user_analysis.lazy().select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users")).collect())
share_per_mean = (user_analysis.lazy().select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users")).collect())

In [71]:
concat = pl.concat([xact_per_mean,share_per_mean,ratio_xact_share_per],how="horizontal")
concat

mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.10943,0.151174,0.723865


In [73]:
ram_ratio = (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
        .select((pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")))
ram_mean_xact_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"))
ram_mean_share_of_users=user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list)).select(pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"))
ram = (
    pl.concat([ram_mean_xact_of_users, ram_mean_share_of_users, ram_ratio], how="horizontal").collect())
ram


mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
f64,f64,f64
0.146889,0.158311,0.927851


In [ ]:
xact_share=(user_analysis.lazy()
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)
).collect()


msg,xact,share
str,f64,f64
"""00003""",95.558173,126.844615
"""00013""",94.809351,103.805721
"""00023""",92.521947,83.708245
"""00033""",89.172332,67.710625
"""00103""",86.151538,103.092428
…,…,…
"""33320""",16.694351,27.341851
"""33330""",16.447957,21.235721
"""23300""",16.440457,48.872668


In [94]:
xact_share_mean= xact_share.select(pl.col("xact").mean(),pl.col("share").mean())
xact_share_ratio=xact_share.select([(pl.col("xact").mean()/pl.col("share").mean()).alias("xact/share")])
combine1 = pl.concat([xact_share_mean,xact_share_ratio],how="horizontal")
combine1

xact,share,xact/share
f64,f64,f64
40.865631,56.454788,0.723865


In [83]:
ram_xact_share= (user_analysis.lazy().filter(pl.col("msg").is_in(ram_msg_list))
.group_by(["ip","us_id","msg"]).agg(pl.col("num_xact_of_users").sum().alias("xact"),pl.col("num_share_of_users").sum().alias("share"))
.group_by("msg").agg(pl.col("xact").mean(),pl.col("share").mean()).sort("xact",descending = True)
).collect()
ram_xact_share

msg,xact,share
str,f64,f64
"""00003""",95.558173,126.844615
"""00013""",94.809351,103.805721
"""00023""",92.521947,83.708245
"""00033""",89.172332,67.710625
"""00103""",86.151538,103.092428
…,…,…
"""20023""",74.801082,61.905841
"""00112""",74.621106,99.429663
"""00122""",74.325216,81.529712


In [92]:
ram_xact_share_mean= ram_xact_share.select(pl.col("xact").mean(),pl.col("share").mean())
ram_xact_share_ratio=ram_xact_share.select([(pl.col("xact").mean()/pl.col("share").mean()).alias("xact/share")])
combine = pl.concat([ram_xact_share_mean,ram_xact_share_ratio],how="horizontal")
combine

xact,share,xact/share
f64,f64,f64
80.890839,87.180893,0.927851


In [79]:
ram_ratio_mean = ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
ram_ratio_mean

ColumnNotFoundError: mean_num_xact

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'select' <---
SELECT [[(col("num_xact_of_users").mean()) / (col("num_share_of_users").mean())].alias("xact_share_ratio")]
FROM
  FILTER col("msg").is_in([["00003", "00013", … "10012"]])
  FROM
    DF ["ip", "us_id", "msg", "user_id", ...]; PROJECT */7 COLUMNS

In [ ]:
re_ram_msg = pl.read_parquet("../../experiments/twitter_81306/reverse_msg_us_an.parquet")
action_stats_per_msg = (re_ram_msg.lazy().filter(pl.col("msg").is_in(reverse_msg_list)).group_by("msg").agg(
    pl.col("num_xact_of_users").mean(),
    pl.col("num_share_of_users").mean())
)
ram =lf_rob_details.join(action_stats_per_msg.collect(),left_on="msg", right_on="msg").sort(pl.col("mean_num_xact"), descending=True)
re_ram_ratio = ram.lazy().select(pl.col("msg"),pl.col("mean_num_xact"),pl.col("num_xact_of_users"),pl.col("num_share_of_users"),(pl.col("num_xact_of_users") / pl.col("num_share_of_users")).alias("xact_share_ratio")).collect()
re_ram_ratio

msg,mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
str,f64,f64,f64,f64
"""13333""",176.582812,0.200084,0.208078,0.96158
"""02233""",175.57825,0.114405,0.280044,0.408525
"""01332""",171.856437,0.101695,0.294872,0.344879
"""01233""",169.0435,0.08959,0.292721,0.306059
"""00332""",168.221312,0.078627,0.307673,0.255555
"""00233""",163.912813,0.069287,0.303262,0.228472
"""02333""",144.857687,0.12186,0.269813,0.451648
"""03333""",142.906688,0.151186,0.257636,0.586821
"""01333""",139.436688,0.094801,0.281407,0.336882


['30000',
 '31000',
 '32000',
 '30001',
 '20000',
 '22000',
 '21000',
 '33000',
 '31001',
 '30100']

In [46]:
re_ram_ratio_mean = re_ram_ratio.select(pl.col("mean_num_xact").mean(),pl.col("num_xact_of_users").mean(),pl.col("num_share_of_users").mean(),pl.col("xact_share_ratio").mean())
re_ram_ratio_mean

mean_num_xact,num_xact_of_users,num_share_of_users,xact_share_ratio
f64,f64,f64,f64
159.080519,0.109499,0.279096,0.412902


逆の順番に情報を提供すると

In [16]:
re_ram_xact_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_xact_of_users").mean())
re_ram_share_mean= user_analysis.lazy().filter(pl.col("msg") == "00033").select(pl.col("num_share_of_users").mean())

re_ram_xact_mean.collect()
re_ram_share_mean.collect()

num_share_of_users
f64
0.315331


最適戦略

In [5]:
opt_ratio = (opt_ratio_lazy.lazy().group_by(["msg","us_id","ip"])
        .agg(pl.col("num_xact_of_users").mean().alias("mean_num_xact_of_users"),
             pl.col("num_share_of_users").mean().alias("mean_num_share_of_users"),
            (pl.col("num_xact_of_users").mean() / pl.col("num_share_of_users").mean()).alias("xact_share_ratio")
        )
        .collect())
opt_ratio

msg,us_id,ip,mean_num_xact_of_users,mean_num_share_of_users,xact_share_ratio
str,u64,u64,f64,f64,f64
"""30010""",13,79973,0.134906,0.148113,0.910828
"""30000""",5,69023,0.06342,0.051303,1.23619
"""30000""",4,79973,0.047638,0.054216,0.878664
"""30002""",13,12605,0.02523,0.023032,1.095413
"""32221""",3,33649,0.45,0.71,0.633803
…,…,…,…,…,…
"""21113""",8,68985,0.226,0.247333,0.913747
"""33000""",11,38598,0.388505,0.373066,1.041383
"""30103""",9,9199,0.057205,0.092773,0.61661
